# Option 1 Capacity-Sanity Stress Test

This notebook tests the alternative instance strategy discussed for Gurobi stress testing: sample jobs from Alibaba-calibrated empirical rows, assign arrival/deadline windows directly, run necessary capacity sanity checks, and then let Gurobi determine whether the instance is feasible and how hard it is to solve.

This intentionally does **not** use the feasible-by-construction generator. The purpose is to explore the boundary between plausible aggregate demand and actual schedulability under time-window, GPU-type, GPU-count, CPU, memory, power, PUE, renewable, and contracted-power constraints.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys
import time
from typing import Any

import numpy as np
import pandas as pd
from gurobipy import GRB

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import ModelConfig
from src.data.scenarios import DEFAULT_GPU_POWER_KW, build_alibaba_gpu_type_clusters
from src.data.validation import validate_clusters, validate_hourly_inputs, validate_jobs
from src.evaluation.metrics import compute_summary_metrics
from src.evaluation.results import extract_cluster_hourly_results, extract_hourly_results, extract_schedule
from src.instance_generator.alibaba_calibration import AlibabaCalibrationConfig, build_alibaba_generator_calibration
from src.milp.gurobi_model import build_milp_model
from src.milp.solve import solve_model


## Experiment Configuration

The workload scale is controlled only by `JOB_COUNTS` and the sampled Alibaba rows. There is no target-utilization knob and no hidden greedy placement. The capacity checks below are diagnostic: they can identify cases that are impossible in aggregate, but they cannot prove feasibility because time windows and GPU-type restrictions still matter.


In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "experiments" / "option1_capacity_sanity_stress"
BASE_CALIBRATION_DIR = PROJECT_ROOT / "experiments" / "outputs" / "alibaba_2023_eda"
RAW_ALIBABA_DIR = PROJECT_ROOT / "data" / "raw" / "alibaba_gpu_v2023"

SLOT_MINUTES = 15
HORIZON_HOURS = 24
HORIZON_SLOTS = int(HORIZON_HOURS * 60 / SLOT_MINUTES)
DELTA_T = SLOT_MINUTES / 60
MAX_RUNTIME_HOURS_FOR_CALIBRATION = 4.0

JOB_COUNTS = list(range(40_000, 70_001, 5_000))
SEEDS = list(range(11, 31))  # 20 instances per job count.

# Debug override example:
# JOB_COUNTS = [1_000]
# SEEDS = [11]

TIME_LIMIT_SECONDS = 120
MIP_GAP = 0.01

PUE = 1.20
RENEWABLE_PRICE = 40.0
PEAK_PRICE = 1_000.0
BASELINE_LOAD_MW = 0.0

# Waiting-window model: latest_start = arrival_slot + sampled Alibaba wait * multiplier,
# clipped so the job still finishes inside the horizon. This gives some jobs much more
# flexibility than others while preserving the empirical wait-time scale.
WAIT_MULTIPLIERS = np.array([1, 2, 4, 8, 16, 32], dtype=int)
WAIT_MULTIPLIER_PROBABILITIES = np.array([0.30, 0.25, 0.20, 0.15, 0.07, 0.03])
MIN_WAIT_SLOTS = 1
MAX_WAIT_SLOTS = HORIZON_SLOTS - 1

GPU_POWER_KW = DEFAULT_GPU_POWER_KW.copy()
POWER_JITTER_FRACTION = 0.0

ENFORCE_GPU_CONSTRAINTS = True
ENFORCE_CPU_CONSTRAINTS = True
ENFORCE_MEMORY_CONSTRAINTS = True

SAVE_INPUT_TABLES = False
SAVE_SOLUTION_TABLES = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Calibration and Scenario Inputs

The calibration table is regenerated for the selected slot size if needed. For 15-minute slots, the notebook expects durations in 15-minute units; using 60-minute calibration rows would distort both duration and feasible-start counts.


In [ ]:
def calibration_dir_for_slot(slot_minutes: int) -> Path:
    """Return the calibration output directory associated with a slot size."""

    if slot_minutes == 60:
        return BASE_CALIBRATION_DIR
    return BASE_CALIBRATION_DIR.with_name(f"{BASE_CALIBRATION_DIR.name}_{slot_minutes}min")


def ensure_alibaba_calibration(slot_minutes: int) -> Path:
    """Build Alibaba calibration outputs when missing or using stale slot metadata."""

    calibration_dir = calibration_dir_for_slot(slot_minutes)
    samples_path = calibration_dir / "alibaba_generator_job_samples.csv"
    needs_rebuild = not samples_path.exists()
    if samples_path.exists():
        existing = pd.read_csv(samples_path, nrows=100)
        needs_rebuild = "slot_minutes" not in existing.columns or set(existing["slot_minutes"].dropna().astype(int)) != {slot_minutes}
    if needs_rebuild:
        build_alibaba_generator_calibration(
            AlibabaCalibrationConfig(
                raw_dir=RAW_ALIBABA_DIR,
                output_dir=calibration_dir,
                max_runtime_hours=MAX_RUNTIME_HOURS_FOR_CALIBRATION,
                slot_minutes=slot_minutes,
            )
        )
    return calibration_dir


def load_node_capacity(path: Path) -> tuple[dict[str, float], dict[str, float]]:
    """Load aggregate CPU and memory capacities by Alibaba GPU type."""

    capacity_df = pd.read_csv(path)
    cpu_capacity = dict(zip(capacity_df["gpu_type"], capacity_df["cpu_capacity"], strict=True))
    memory_capacity = dict(zip(capacity_df["gpu_type"], capacity_df["memory_capacity_gb"], strict=True))
    return cpu_capacity, memory_capacity


CALIBRATION_DIR = ensure_alibaba_calibration(SLOT_MINUTES)
SAMPLES_PATH = CALIBRATION_DIR / "alibaba_generator_job_samples.csv"
CLUSTER_CAPACITY_PATH = CALIBRATION_DIR / "alibaba_cluster_capacity_from_nodes.csv"

samples_df = pd.read_csv(SAMPLES_PATH)
cpu_capacity, memory_capacity = load_node_capacity(CLUSTER_CAPACITY_PATH)
clusters_df = build_alibaba_gpu_type_clusters(
    gpu_power_kw=GPU_POWER_KW,
    cpu_capacity=cpu_capacity,
    memory_capacity_gb=memory_capacity,
)

TOTAL_IT_CAPACITY_MW = float(clusters_df["capacity"].sum())
CONTRACTED_POWER_MW = 0.75 * TOTAL_IT_CAPACITY_MW * PUE

validate_clusters(clusters_df)
clusters_df


## Energy Inputs

Renewable availability is scenario data, not part of the feasibility construction. The sinusoidal profile below is intentionally simple and should later be replaced by a calibrated renewable scenario once those ranges are agreed.


In [ ]:
def build_hourly_inputs() -> pd.DataFrame:
    """Create slot-level renewable availability, grid price, baseline load, and PUE inputs."""

    slots = np.arange(HORIZON_SLOTS)
    hour_of_day = slots * DELTA_T
    solar_shape = np.maximum(0.0, np.sin(np.pi * (hour_of_day - 6.0) / 12.0))
    renewable_available = 0.35 * TOTAL_IT_CAPACITY_MW * PUE * solar_shape
    grid_price = 70.0 + 25.0 * ((hour_of_day >= 18.0) & (hour_of_day <= 22.0)).astype(float)
    grid_price = grid_price - 10.0 * ((hour_of_day >= 1.0) & (hour_of_day <= 5.0)).astype(float)
    hourly_df = pd.DataFrame(
        {
            "hour": slots.astype(int),
            "renewable_available": renewable_available.astype(float),
            "grid_price": grid_price.astype(float),
            "baseline_load": BASELINE_LOAD_MW,
            "pue": PUE,
        }
    )
    validate_hourly_inputs(hourly_df)
    return hourly_df


hourly_df = build_hourly_inputs()
hourly_df.head(), hourly_df.tail()


## Replay-Like Instance Builder

This builder samples empirical Alibaba rows with replacement, keeps duration/GPU/CPU/memory correlations from the trace, then creates an arrival and deadline window. The window is not guaranteed to be schedulable; that is exactly what this experiment is testing.


In [ ]:
SECONDS_PER_DAY = 24 * 60 * 60
SLOT_SECONDS = SLOT_MINUTES * 60


def normalize_gpu_requirement(value: Any) -> str:
    """Normalize a GPU-type requirement string without inventing workload categories."""

    if pd.isna(value) or str(value).strip() in {"", "None", "nan"}:
        return ""
    tokens = [token.strip() for token in str(value).split("|") if token.strip()]
    return "|".join(sorted(dict.fromkeys(tokens)))


def allowed_gpu_types(value: Any, available_gpu_types: set[str]) -> set[str]:
    """Return the physical GPU types that can run a sampled job."""

    normalized = normalize_gpu_requirement(value)
    if not normalized:
        return set(available_gpu_types)
    return {token for token in normalized.split("|") if token in available_gpu_types}


def job_has_individual_compatible_cluster(row: pd.Series, clusters: pd.DataFrame) -> bool:
    """Check whether one sampled job can fit on at least one cluster in isolation."""

    allowed = allowed_gpu_types(row["gpu_type_required"], set(clusters["gpu_type"]))
    if not allowed:
        return False
    for cluster in clusters.itertuples(index=False):
        if cluster.gpu_type not in allowed:
            continue
        if int(row["gpu_count_required"]) > int(cluster.gpu_count):
            continue
        if float(row["cpu_required"]) > float(cluster.cpu_capacity):
            continue
        if float(row["memory_required_gb"]) > float(cluster.memory_capacity_gb):
            continue
        return True
    return False


def per_gpu_power_kw_for_requirement(value: Any, clusters: pd.DataFrame) -> float:
    """Estimate per-GPU IT power for a job from its compatible GPU types."""

    available = set(clusters["gpu_type"])
    allowed = allowed_gpu_types(value, available)
    if not allowed:
        raise ValueError(f"job has no available GPU type for requirement {value!r}")
    weighted = clusters[clusters["gpu_type"].isin(allowed)].copy()
    weighted["gpu_power_kw"] = weighted["gpu_type"].map(GPU_POWER_KW).astype(float)
    return float(np.average(weighted["gpu_power_kw"], weights=weighted["gpu_count"]))


def sample_arrival_slot(row: pd.Series, duration_slots: int, rng: np.random.Generator) -> int:
    """Sample an arrival slot from Alibaba creation time modulo the 24-hour horizon."""

    latest_possible_start = HORIZON_SLOTS - duration_slots
    if latest_possible_start < 0:
        raise ValueError("sampled job duration exceeds the horizon")
    if "creation_time" in row and pd.notna(row["creation_time"]):
        arrival = int((float(row["creation_time"]) % SECONDS_PER_DAY) // SLOT_SECONDS)
        if arrival <= latest_possible_start:
            return arrival
    return int(rng.integers(0, latest_possible_start + 1))


def sample_wait_window_slots(row: pd.Series, rng: np.random.Generator) -> tuple[int, int]:
    """Sample allowed waiting slots and the multiplier used to create the window."""

    raw_wait_hours = max(0.0, float(row.get("wait_hours", 0.0)))
    base_wait_slots = max(MIN_WAIT_SLOTS, int(np.ceil(raw_wait_hours * 60.0 / SLOT_MINUTES)))
    multiplier = int(rng.choice(WAIT_MULTIPLIERS, p=WAIT_MULTIPLIER_PROBABILITIES))
    wait_slots = int(np.clip(base_wait_slots * multiplier, MIN_WAIT_SLOTS, MAX_WAIT_SLOTS))
    return wait_slots, multiplier


def build_replay_like_jobs(num_jobs: int, seed: int, samples: pd.DataFrame, clusters: pd.DataFrame) -> pd.DataFrame:
    """Build one sampled Alibaba-like job table without hidden feasible placement."""

    rng = np.random.default_rng(seed)
    eligible = samples.copy()
    eligible["gpu_type_required"] = eligible["gpu_type_required"].map(normalize_gpu_requirement)
    eligible = eligible[eligible["duration"].astype(int) <= HORIZON_SLOTS].copy()
    eligible = eligible[eligible.apply(lambda row: job_has_individual_compatible_cluster(row, clusters), axis=1)]
    if eligible.empty:
        raise ValueError("no Alibaba samples can individually fit the configured clusters")

    sampled_positions = rng.integers(0, len(eligible), size=num_jobs)
    sampled = eligible.iloc[sampled_positions].reset_index(drop=True)

    rows = []
    for idx, row in sampled.iterrows():
        duration = int(row["duration"])
        arrival = sample_arrival_slot(row, duration, rng)
        wait_slots, wait_multiplier = sample_wait_window_slots(row, rng)
        latest_possible_start = HORIZON_SLOTS - duration
        latest_start = min(latest_possible_start, arrival + wait_slots)
        gpu_count = int(row["gpu_count_required"])
        per_gpu_power_kw = per_gpu_power_kw_for_requirement(row["gpu_type_required"], clusters)
        power_mw = gpu_count * per_gpu_power_kw / 1000.0
        if POWER_JITTER_FRACTION > 0:
            jitter = rng.uniform(1.0 - POWER_JITTER_FRACTION, 1.0 + POWER_JITTER_FRACTION)
            power_mw *= jitter
        rows.append(
            {
                "job_id": f"job_{idx:06d}",
                "category": "gpu_unrestricted" if row["gpu_type_required"] == "" else "gpu_restricted",
                "workload_family": "alibaba_replay_like",
                "duration": duration,
                "power": float(power_mw),
                "earliest_start": int(arrival),
                "latest_start": int(latest_start),
                "gpu_type_required": row["gpu_type_required"],
                "gpu_count_required": gpu_count,
                "cpu_required": float(row["cpu_required"]),
                "memory_required_gb": float(row["memory_required_gb"]),
                "runtime_hours_sampled": float(row["runtime_hours"]),
                "wait_hours_sampled": float(row.get("wait_hours", 0.0)),
                "wait_multiplier": wait_multiplier,
                "allowed_wait_slots": int(latest_start - arrival),
            }
        )
    jobs_df = pd.DataFrame(rows)
    validate_jobs(jobs_df)
    return jobs_df


## Necessary Capacity Checks

These checks answer: “Is the total amount of work obviously above the physical capacity of the fleet?” They are not sufficient because a workload can pass aggregate checks but still be infeasible due to arrivals, deadlines, and GPU-type bottlenecks.


In [ ]:
def capacity_sanity_checks(jobs_df: pd.DataFrame, clusters: pd.DataFrame) -> dict[str, float]:
    """Compute necessary aggregate resource-demand ratios for one sampled instance."""

    horizon_hours = HORIZON_SLOTS * DELTA_T
    total_power_capacity_mwh = float((clusters["capacity"] * horizon_hours).sum())
    total_gpu_capacity_hours = float((clusters["gpu_count"] * horizon_hours).sum())
    total_cpu_capacity_hours = float((clusters["cpu_capacity"] * horizon_hours).sum())
    total_memory_capacity_gbh = float((clusters["memory_capacity_gb"] * horizon_hours).sum())

    job_it_energy_mwh = float((jobs_df["power"] * jobs_df["duration"] * DELTA_T).sum())
    job_gpu_hours = float((jobs_df["gpu_count_required"] * jobs_df["duration"] * DELTA_T).sum())
    job_cpu_hours = float((jobs_df["cpu_required"] * jobs_df["duration"] * DELTA_T).sum())
    job_memory_gbh = float((jobs_df["memory_required_gb"] * jobs_df["duration"] * DELTA_T).sum())

    single_type_jobs = jobs_df[jobs_df["gpu_type_required"].astype(str).str.contains("|", regex=False) == False].copy()
    single_type_jobs = single_type_jobs[single_type_jobs["gpu_type_required"].astype(str).str.len() > 0]
    gpu_ratios_by_type = []
    for cluster in clusters.itertuples(index=False):
        required = single_type_jobs[single_type_jobs["gpu_type_required"] == cluster.gpu_type]
        demand = float((required["gpu_count_required"] * required["duration"] * DELTA_T).sum())
        capacity = float(cluster.gpu_count) * horizon_hours
        gpu_ratios_by_type.append(demand / capacity if capacity > 0 else np.nan)

    window_width = jobs_df["latest_start"] - jobs_df["earliest_start"] + 1
    return {
        "job_it_energy_mwh": job_it_energy_mwh,
        "global_power_demand_ratio": job_it_energy_mwh / total_power_capacity_mwh,
        "global_gpu_demand_ratio": job_gpu_hours / total_gpu_capacity_hours,
        "global_cpu_demand_ratio": job_cpu_hours / total_cpu_capacity_hours,
        "global_memory_demand_ratio": job_memory_gbh / total_memory_capacity_gbh,
        "max_single_gpu_type_demand_ratio": float(np.nanmax(gpu_ratios_by_type)) if gpu_ratios_by_type else 0.0,
        "median_window_width_slots": float(window_width.median()),
        "p10_window_width_slots": float(window_width.quantile(0.10)),
        "p90_window_width_slots": float(window_width.quantile(0.90)),
        "unrestricted_gpu_share": float((jobs_df["gpu_type_required"].astype(str).str.len() == 0).mean()),
        "mean_duration_slots": float(jobs_df["duration"].mean()),
    }


def summarize_cluster_utilization(cluster_results: pd.DataFrame) -> dict[str, float]:
    """Summarize solved resource utilization across all clusters and slots."""

    metrics: dict[str, float] = {}
    utilization_specs = {
        "power": ("cluster_load", "capacity"),
        "gpu": ("cluster_gpu_load", "gpu_capacity"),
        "cpu": ("cluster_cpu_load", "cpu_capacity"),
        "memory": ("cluster_memory_load", "memory_capacity_gb"),
    }
    for label, (load_col, cap_col) in utilization_specs.items():
        if load_col not in cluster_results.columns or cap_col not in cluster_results.columns:
            continue
        denominator = cluster_results[cap_col].replace(0, np.nan)
        utilization = (cluster_results[load_col] / denominator).replace([np.inf, -np.inf], np.nan).fillna(0.0)
        metrics[f"max_{label}_utilization"] = float(utilization.max())
        metrics[f"mean_{label}_utilization"] = float(utilization.mean())
    return metrics


## Solve One Instance

The function below generates the sampled jobs, computes diagnostic capacity ratios, builds the MILP, solves it, and extracts compact metrics. Detailed input and solution tables can be saved by toggling `SAVE_INPUT_TABLES` and `SAVE_SOLUTION_TABLES`.


In [ ]:
STATUS_NAMES = {
    GRB.OPTIMAL: "optimal",
    GRB.TIME_LIMIT: "time_limit",
    GRB.SUBOPTIMAL: "suboptimal",
    GRB.INFEASIBLE: "infeasible",
    GRB.INF_OR_UNBD: "infeasible_or_unbounded",
    GRB.UNBOUNDED: "unbounded",
}


def safe_model_attr(model: Any, attr: str, default: float | None = None) -> float | None:
    """Return a Gurobi attribute when available after optimization."""

    try:
        return float(getattr(model, attr))
    except Exception:
        return default


def solve_option1_instance(num_jobs: int, seed: int) -> dict[str, Any]:
    """Generate and solve one Option 1 sampled instance, returning summary metrics."""

    run_name = f"jobs_{num_jobs:06d}_seed_{seed}"
    run_dir = OUTPUT_DIR / run_name
    result: dict[str, Any] = {
        "num_jobs": num_jobs,
        "seed": seed,
        "run_name": run_name,
    }

    try:
        if SAVE_INPUT_TABLES or SAVE_SOLUTION_TABLES:
            run_dir.mkdir(parents=True, exist_ok=True)

        generation_start = time.time()
        jobs_df = build_replay_like_jobs(num_jobs, seed, samples_df, clusters_df)
        sanity = capacity_sanity_checks(jobs_df, clusters_df)
        generation_s = time.time() - generation_start
        result.update({"generation_s": generation_s, **sanity})

        if SAVE_INPUT_TABLES:
            jobs_df.to_csv(run_dir / "jobs.csv", index=False)
            hourly_df.to_csv(run_dir / "hourly.csv", index=False)
            clusters_df.to_csv(run_dir / "clusters.csv", index=False)
            (run_dir / "capacity_sanity.json").write_text(json.dumps(sanity, indent=2))

        config = ModelConfig(
            contracted_power=CONTRACTED_POWER_MW,
            renewable_price=RENEWABLE_PRICE,
            peak_price=PEAK_PRICE,
            delta_t=DELTA_T,
            pue=PUE,
        )

        build_start = time.time()
        model, variables = build_milp_model(
            jobs_df,
            hourly_df,
            clusters_df,
            config,
            model_name=f"option1_{run_name}",
            enforce_gpu_constraints=ENFORCE_GPU_CONSTRAINTS,
            enforce_cpu_constraints=ENFORCE_CPU_CONSTRAINTS,
            enforce_memory_constraints=ENFORCE_MEMORY_CONSTRAINTS,
        )
        build_s = time.time() - build_start
        solve_start = time.time()
        model = solve_model(model, time_limit=TIME_LIMIT_SECONDS, mip_gap=MIP_GAP)
        solve_s = time.time() - solve_start

        hourly_results = extract_hourly_results(hourly_df, variables)
        cluster_results = extract_cluster_hourly_results(variables)
        summary = compute_summary_metrics(hourly_results, config)
        utilization = summarize_cluster_utilization(cluster_results)

        if SAVE_SOLUTION_TABLES:
            extract_schedule(jobs_df, variables).to_csv(run_dir / "schedule.csv", index=False)
            hourly_results.to_csv(run_dir / "hourly_results.csv", index=False)
            cluster_results.to_csv(run_dir / "cluster_hourly_results.csv", index=False)

        result.update(
            {
                "status": "solved_or_feasible",
                "gurobi_status": STATUS_NAMES.get(model.Status, str(model.Status)),
                "build_s": build_s,
                "solve_s": solve_s,
                "runtime_s": safe_model_attr(model, "Runtime"),
                "mip_gap": safe_model_attr(model, "MIPGap"),
                "objective": safe_model_attr(model, "ObjVal"),
                "best_bound": safe_model_attr(model, "ObjBound"),
                "assignment_vars": len(variables["x"]),
                "num_vars": int(model.NumVars),
                "num_constraints": int(model.NumConstrs),
                **summary,
                **utilization,
            }
        )
    except Exception as exc:
        result.update(
            {
                "status": "failed",
                "error_type": type(exc).__name__,
                "error": str(exc),
            }
        )
    return result


## Run Stress Grid

This cell runs 7 job-count levels times 20 seeds = 140 solves. It is expected to be slow near the feasibility and memory boundary. The resulting CSV is append-free and reproducible from the settings above.


In [ ]:
rows = []
for num_jobs in JOB_COUNTS:
    for seed in SEEDS:
        print(f"Solving Option 1 instance: jobs={num_jobs}, seed={seed}")
        result = solve_option1_instance(num_jobs, seed)
        rows.append(result)
        print(
            {
                key: result.get(key)
                for key in [
                    "status",
                    "gurobi_status",
                    "runtime_s",
                    "mip_gap",
                    "assignment_vars",
                    "global_gpu_demand_ratio",
                    "global_cpu_demand_ratio",
                    "global_memory_demand_ratio",
                    "max_gpu_utilization",
                    "max_cpu_utilization",
                    "max_memory_utilization",
                ]
            }
        )

results_df = pd.DataFrame(rows)
expected_numeric_columns = [
    "runtime_s",
    "mip_gap",
    "assignment_vars",
    "global_gpu_demand_ratio",
    "global_cpu_demand_ratio",
    "global_memory_demand_ratio",
    "max_gpu_utilization",
    "max_cpu_utilization",
    "max_memory_utilization",
]
for column in expected_numeric_columns:
    if column not in results_df.columns:
        results_df[column] = np.nan
results_path = OUTPUT_DIR / "option1_capacity_sanity_results.csv"
results_df.to_csv(results_path, index=False)
results_path, results_df.head()


## Aggregate Results

The aggregation separates generation/build/solve behavior from actual resource pressure. A useful pattern is: aggregate demand ratios close to or below 1, but solve failures or infeasibility still appearing as windows become tight or GPU-type bottlenecks dominate.


In [ ]:
if results_df.empty:
    raise ValueError("Run the stress grid first.")

agg = (
    results_df.assign(solved=lambda df: df["status"].eq("solved_or_feasible"))
    .groupby("num_jobs", dropna=False)
    .agg(
        instances=("seed", "count"),
        solved_instances=("solved", "sum"),
        median_runtime_s=("runtime_s", "median"),
        p90_runtime_s=("runtime_s", lambda s: s.dropna().quantile(0.90) if s.notna().any() else np.nan),
        median_mip_gap=("mip_gap", "median"),
        median_assignment_vars=("assignment_vars", "median"),
        median_global_gpu_ratio=("global_gpu_demand_ratio", "median"),
        median_global_cpu_ratio=("global_cpu_demand_ratio", "median"),
        median_global_memory_ratio=("global_memory_demand_ratio", "median"),
        median_max_gpu_utilization=("max_gpu_utilization", "median"),
        median_max_cpu_utilization=("max_cpu_utilization", "median"),
        median_max_memory_utilization=("max_memory_utilization", "median"),
        median_window_width_slots=("median_window_width_slots", "median"),
    )
    .reset_index()
)
agg["solve_rate"] = agg["solved_instances"] / agg["instances"]
agg.to_csv(OUTPUT_DIR / "option1_capacity_sanity_aggregate.csv", index=False)
agg


## Scaling Plots


In [ ]:
import matplotlib.pyplot as plt

if not results_df.empty:
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))

    axes[0, 0].plot(agg["num_jobs"], agg["solve_rate"], marker="o")
    axes[0, 0].set_title("Solve Rate")
    axes[0, 0].set_xlabel("Jobs")
    axes[0, 0].set_ylabel("Solved / generated instances")
    axes[0, 0].set_ylim(-0.05, 1.05)

    axes[0, 1].plot(agg["num_jobs"], agg["median_runtime_s"], marker="o", label="median")
    axes[0, 1].plot(agg["num_jobs"], agg["p90_runtime_s"], marker="o", label="p90")
    axes[0, 1].set_title("Gurobi Runtime")
    axes[0, 1].set_xlabel("Jobs")
    axes[0, 1].set_ylabel("Seconds")
    axes[0, 1].legend()

    axes[1, 0].plot(agg["num_jobs"], agg["median_assignment_vars"], marker="o")
    axes[1, 0].set_title("Assignment Variables")
    axes[1, 0].set_xlabel("Jobs")
    axes[1, 0].set_ylabel("x[j,k,s] count")

    axes[1, 1].plot(agg["num_jobs"], agg["median_mip_gap"], marker="o")
    axes[1, 1].set_title("Median MIP Gap")
    axes[1, 1].set_xlabel("Jobs")
    axes[1, 1].set_ylabel("Relative optimality gap")

    plt.tight_layout()
    plt.show()


In [ ]:
import matplotlib.pyplot as plt

if not results_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].plot(agg["num_jobs"], agg["median_global_gpu_ratio"], marker="o", label="GPU")
    axes[0].plot(agg["num_jobs"], agg["median_global_cpu_ratio"], marker="o", label="CPU")
    axes[0].plot(agg["num_jobs"], agg["median_global_memory_ratio"], marker="o", label="Memory")
    axes[0].axhline(1.0, color="black", linestyle="--", linewidth=1)
    axes[0].set_title("Necessary Aggregate Demand Ratios")
    axes[0].set_xlabel("Jobs")
    axes[0].set_ylabel("Demand / horizon capacity")
    axes[0].legend()

    axes[1].plot(agg["num_jobs"], agg["median_max_gpu_utilization"], marker="o", label="GPU")
    axes[1].plot(agg["num_jobs"], agg["median_max_cpu_utilization"], marker="o", label="CPU")
    axes[1].plot(agg["num_jobs"], agg["median_max_memory_utilization"], marker="o", label="Memory")
    axes[1].axhline(1.0, color="black", linestyle="--", linewidth=1)
    axes[1].set_title("Solved Peak Utilization")
    axes[1].set_xlabel("Jobs")
    axes[1].set_ylabel("Peak utilization")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


## Inspect Failure Modes

This table helps separate model-building or memory issues from true Gurobi infeasibility/no-solution outcomes.


In [ ]:
if "error_type" not in results_df.columns:
    failure_summary = pd.DataFrame(columns=["num_jobs", "error_type", "error", "count"])
else:
    failure_summary = (
        results_df[results_df["status"] != "solved_or_feasible"]
        .groupby(["num_jobs", "error_type", "error"], dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values(["num_jobs", "count"], ascending=[True, False])
    )
failure_summary.head(30)


## Notes on `MIP_GAP`

`MIP_GAP` is the relative optimality gap passed to Gurobi as `model.Params.MIPGap`. With `MIP_GAP = 0.01`, Gurobi may stop once the best feasible solution is within 1% of the best known mathematical bound. This is not a feasibility tolerance. A solution with `mip_gap = 0.01` is feasible, but not proven globally optimal beyond that 1% bound.
